# 2.2 Colonisation time estimate: days to colonise an oyster bag from group, substrate and temperature

**Question:** can the group (Cool, Tropical or King oyster), the substrate group and the temperature predict the days to colonise better than the 2.1 baseline (the group's median range)?

Scope agreed 10-15: oysters (Pleurotus) only. Inputs are the plot group, the substrate group (straw, supplemented sawdust, compost, cotton waste, other) and the temperature. **Spawn % is not an input**: it only triggers a warning outside 5-20% of the total mix. C:N (`cn_mix_estimated`) is fitted with and without and kept only if it clearly helps. Agaricus bisporus is excluded (10-15). Agar data never trains.

**Data**
- Guide values: the Pleurotus rows of `MycoMaterials/germini/colonisation_consensus.csv`, parsed exactly as in 2.1 (10 book rows, 5 usemycro rows). The usemycro rows give only a day range (no typical day, no temperature, no substrate), so they cannot enter the regression; they feed the 2.1 band and are used in the plausibility check at the end.
- Measurements: real substrate bags from our extracted tables, only where the species is Pleurotus and the table gives days to full colonisation (section 1 lists what was checked and why each set is in or out).

**Model:** linear regression on log(typical days) = group + substrate group + temperature term. Straw and Cool oyster are the reference, so every effect reads as a plain % change in days. Min and max days come from the typical prediction times the median min/typ and max/typ ratios of the data.

**Check:** leave one source out (a book, or one paper, is never on both sides), error in days against the 2.1 baseline (the group median, also given the same training rows). The notebook prints a plain verdict. If the model is not better, the app keeps the 2.1 band.

**Warning about the size of this data set.** About 30 rows, 11 sources, and most of the temperatures sit between 21 and 28 °C. Read every coefficient as rough. Nothing is written to disk by the model part; the coefficients file is only written after Sylas says yes (`WRITE_COEFFICIENTS`).

Sections: 0 tidy the guide rows (2.1 parsing), 1 measured rows, 2 choose the model (temperature form, C:N), 3 coefficients as % effects and the band, 4 check against the 2.1 baseline and verdict, 5 history, 6 output for the app.

In [ ]:
# EDIT HERE: the choices for this notebook
MEASURED_SETS = ["d025", "d026", "d002"]   # extracted datasets with days to full colonisation, Pleurotus only
T_LIMITS = (10.0, 35.0)                    # cardinal limits: "biological range 10-35 C" in the consensus table
TIE_TOLERANCE = 0.03                       # two temperature forms within 3% of each other's error = the same score
CN_MIN_GAIN = 0.10                         # C:N is kept only if it lowers the error by at least 10%
WRITE_COEFFICIENTS = False                 # stays False until Sylas says yes to the file name, folder and contents

In [ ]:
from pathlib import Path
import datetime
import json
import re

import numpy as np
import pandas as pd

from mycodemo.config import INTERIM_DATA_DIR
from mycodemo.growth import colonisation_estimate as ce

# the only place the data paths appear; the consensus file stays outside the repo
DATA_PATH = Path.home() / "Desktop/MycoMaterials/germini/colonisation_consensus.csv"
EXTRACTED = INTERIM_DATA_DIR / "extracted"

pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_rows", 60)
raw = pd.read_csv(DATA_PATH)
raw.shape

## 0. Tidy the guide rows

The parsing is the 2.1 code, copied unchanged (temperature: stated optimum, else the middle of the main range; days: the middle of a range; substrate group by the first-named substrate; plot group by species, or by temperature for "spp." and mixed rows, which 2.1 marks `assigned by temperature`). The 2.1 names are used: Cool oyster, Tropical oyster (the warm species) and King oyster. The mixed rows are the ones assigned by temperature.

In [ ]:
# EDIT HERE: the same choices as 2.1
EXCLUDED = ["Agaricus bisporus"]
PLEUROTUS_GROUPS = {
    "Cool oyster": ["ostreatus"],
    "Tropical oyster": ["pulmonarius", "florida", "sajor-caju", "djamor", "citrinopileatus", "cornucopiae"],
    "King oyster": ["eryngii"],
}
TROPICAL_FROM_C = 26                      # "Pleurotus spp." or mixed rows: this temperature or above = Tropical oyster

In [ ]:
NUM = r"(\d+(?:\.\d+)?)"
RANGE = re.compile(NUM + r"\s*-\s*" + NUM)
SINGLE = re.compile(NUM)


def mid_of(text):
    """Middle of 'a-b', or the value itself for a single number; NaN if no number."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    m = RANGE.search(text)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    m = SINGLE.search(text)
    return float(m.group(1)) if m else np.nan


def parse_temp(text):
    """Return (temp_min, temp_opt, temp_max) in deg C from Incubation_Temp_C text."""
    nan3 = (np.nan, np.nan, np.nan)
    if not isinstance(text, str):
        return nan3
    # stated optimum, looked for in the full text (it can sit inside brackets)
    opt = np.nan
    m = re.search(r"optimum\s+" + NUM + r"(?:\s*-\s*" + NUM + r")?\s*°C", text)
    if m:
        a = float(m.group(1)); b = float(m.group(2)) if m.group(2) else a
        opt = (a + b) / 2
    else:
        m = re.search(NUM + r"\s*°C\s*ideal", text)
        if m:
            opt = float(m.group(1))
    s = re.sub(r"\([^)]*\)", "", text)                      # drop all brackets
    s = re.sub(NUM + r"(\s*-\s*" + NUM + r")?\s*°F", "", s)    # drop F values
    clauses = [c.strip() for c in s.split(";")]
    stop = re.compile(r"injury|damage|cease|threshold|stop", re.I)
    clauses = [c for c in clauses if c and not stop.search(c) and not c.lower().startswith("optimum")]
    if not clauses:
        return nan3
    main = next((c for c in clauses if re.match(r"(Substrate|Core)\b", c, re.I)), clauses[0])
    lo = hi = mid = mid_of(main)
    m = RANGE.search(main)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
    rng = next((c for c in clauses if c.lower().startswith("range")), None)
    if rng and RANGE.search(rng):
        m = RANGE.search(rng)
        lo, hi = float(m.group(1)), float(m.group(2))
    return lo, (opt if not np.isnan(opt) else mid), hi


def parse_spawn(text):
    """Return (low, high, mid) in % of wet substrate."""
    if not isinstance(text, str):
        return (np.nan,) * 3
    s = re.sub(r"\([^)]*\)", "", text).replace("%", "")
    m = RANGE.search(s)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
    else:
        m = SINGLE.search(s)
        if not m:
            return (np.nan,) * 3
        lo = hi = float(m.group(1))
    return lo, hi, (lo + hi) / 2


def substrate_group(text):
    """Group by the first-named substrate (text before the first '+', ',' or ';')."""
    first = re.split(r"\s\+\s|,|;", str(text))[0].lower()
    if "compost" in first:
        return "compost"
    if "cotton" in first:
        return "cotton waste"
    if "sawdust" in first:
        return "supplemented sawdust" if re.search(r"bran|soy|cornmeal", str(text).lower()) else "other"
    if "straw" in first:
        return "straw"
    return "other"


def rough_temp(text):
    """Stated optimum, else the middle of the first range (deg C), else the first number; NaN if none."""
    if not isinstance(text, str):
        return np.nan
    m = re.search(r"optimum\s+" + NUM + r"(?:\s*-\s*" + NUM + r")?", text)
    if m:
        return (float(m.group(1)) + float(m.group(2) if m.group(2) else m.group(1))) / 2
    return mid_of(text)


def plot_group(species, temp_text):
    """(plot_group, group_note). Non-Pleurotus: genus. Pleurotus: PLEUROTUS_GROUPS, or by temperature for spp./mixed rows."""
    genus = species.split()[0]
    if genus != "Pleurotus":
        return genus, ""
    low = species.lower()
    hits = {g for g, words in PLEUROTUS_GROUPS.items() if any(w in low for w in words)}
    if len(hits) == 1 and "spp" not in low:
        return hits.pop(), ""
    t = rough_temp(temp_text)
    return ("Tropical oyster" if t >= TROPICAL_FROM_C else "Cool oyster"), "assigned by temperature"

In [ ]:
is_um = raw["Source_Reference"].str.startswith("usemycro")
guide = pd.DataFrame(index=raw.index)
guide["source_group"] = np.where(is_um, "usemycro", "book")
guide["source"] = raw["Source_Reference"]
guide["species"] = raw["Species"].str.split(r"\s*\(", n=1, regex=True).str[0].str.strip()
guide["label"] = np.where(is_um, raw["Species"], guide["species"] + " #" + raw.index.astype(str))
guide["temp_c"] = [parse_temp(t)[1] for t in raw["Incubation_Temp_C"]]      # temp_opt
guide["days_min"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Min"]]
guide["days_typ"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Typ"]]
guide["days_max"] = [mid_of(str(v)) for v in raw["Colonisation_Days_Max"]]
guide["substrate_group"] = raw["Substrate_Type"].map(substrate_group)
guide["cn"] = raw["cn_mix_estimated"]
pl = [plot_group(s, t) for s, t in zip(raw["Species"], raw["Incubation_Temp_C"])]
guide["plot_group"] = [p[0] for p in pl]
guide["group_note"] = [p[1] for p in pl]
guide["kind"] = np.where(is_um, "usemycro", "book")

pleurotus = raw["Species"].str.startswith("Pleurotus") & ~raw["Species"].str.startswith(tuple(EXCLUDED))
guide = guide[pleurotus]
book = guide[guide["kind"] == "book"].copy()
usemycro = guide[guide["kind"] == "usemycro"].copy()
print(f"{len(book)} book rows and {len(usemycro)} usemycro rows (Pleurotus)")
guide[["kind", "label", "plot_group", "group_note", "substrate_group", "temp_c", "days_min", "days_typ", "days_max", "cn"]]

## 1. Measured rows

Which extracted datasets have **Pleurotus** and **days to full colonisation per recipe**? A row here is one recipe (group mean of several bags). "Full" means 100% surface colonised (d025, d026) or colonisation score 5, the 95-100% class (d002, Musci; the first sampled day with score 5, so an upper bound by the gap between photos). Recipes that never reached score 5 in the 28 days are left out (the real time is longer than the study, so including them at 28 would bias the data fast). The temperature is the incubation temperature stated for the dataset. Substrate group uses the 2.1 rule on the first-named component.

In [ ]:
meas_tables = {n: pd.read_csv(EXTRACTED / f"{n}.csv") for n in ["measurement", "sample", "substrate", "fermentation"]}
m, s, su, f = (meas_tables[k] for k in ["measurement", "sample", "substrate", "fermentation"])

rows, left_out = [], []
for d in MEASURED_SETS:
    x = (m[(m.dataset_id == d) & (m.measurement_type == "mean")]
         .merge(s, on=["sample_id", "dataset_id"]).merge(su, on="substrate_id", how="left")
         .merge(f, on="fermentation_id", how="left"))
    x = x[x["genus"] == "Pleurotus"]
    for sid, g in x.groupby("sample_id", sort=False):
        g = g.sort_values("day_of_growth")
        full = g[(g["surface_colonised_pct"] >= 100) | (g["colonisation_score"] >= 5)]
        if full.empty:
            left_out.append((d, sid, g["plant_name"].iloc[0], "never reached full colonisation in the study"))
            continue
        r = g.iloc[0]
        species = f"Pleurotus {r['species']}"
        rows.append({"kind": "measured", "source": d, "label": sid, "species": species,
                     "plot_group": plot_group(species, str(r["incubation_temp_c"]))[0],
                     "substrate_group": substrate_group(r["plant_name"]), "plant_name": r["plant_name"],
                     "temp_c": r["incubation_temp_c"], "spawn_pct": r["spawn_pct_wt"],
                     "days_typ": full["day_of_growth"].iloc[0], "cn": r["cn_estimate"]})
measured = pd.DataFrame(rows)
print(f"{len(measured)} measured recipes from {measured['source'].nunique()} datasets; left out:")
display(pd.DataFrame(left_out, columns=["dataset", "recipe", "substrate", "why"]))
measured[["source", "label", "plot_group", "substrate_group", "plant_name", "temp_c", "spawn_pct", "days_typ", "cn"]]

In [ ]:
# every dataset checked, and why it is in or out (species and day columns read from the tables)
sp = f.assign(dataset_id=f["fermentation_id"].map(s.drop_duplicates("fermentation_id").set_index("fermentation_id")["dataset_id"]))
sp = sp.groupby("dataset_id")["species"].agg(lambda v: ", ".join(sorted(set(v.dropna()))))
days_cols = m.groupby("dataset_id")[["day_of_growth", "colonisation_score", "surface_colonised_pct"]].count().sum(axis=1)
checked = pd.DataFrame({
    "dataset": ["d003", "d012", "d024", "d025", "d026", "d002 (Musci)", "d001, d021, d027"],
    "species in the tables": [sp.get("d003", "?"), sp.get("d012", "?"), sp.get("d024", "?"), sp.get("d025", "?"),
                              sp.get("d026", "?"), sp.get("d002", "?"), "Pleurotus (growth rates only)"],
    "colonisation readings": [days_cols.get("d003", 0), days_cols.get("d012", 0), days_cols.get("d024", 0),
                              days_cols.get("d025", 0), days_cols.get("d026", 0), days_cols.get("d002", 0), 0],
    "decision": ["out: not Pleurotus", "out: Pleurotus djamor but no colonisation days (flags PSF1-PSF4)",
                 "out: one 'within 18 d' note, no days per recipe", "in", "in", "in (score 5 = 95-100%)",
                 "out: mm/d or cm/d growth rate on tubes or dishes, not days; agar-type data never trains"]})
checked

In [ ]:
data = pd.concat([book.assign(source=book["source"]), measured], ignore_index=True)
data = data[["kind", "source", "label", "plot_group", "substrate_group", "temp_c", "cn", "days_min", "days_typ", "days_max"]]
print(len(data), "rows,", data["source"].nunique(), "sources;", data["kind"].value_counts().to_dict())
display(pd.crosstab(data["plot_group"], data["substrate_group"], margins=True))
print("temperature (C):", data["temp_c"].describe()[["min", "50%", "max"]].round(1).to_dict(),
      "| rows between 21 and 28 C:", int(data["temp_c"].between(21, 28).sum()), "of", len(data))
print("a substrate group with no rows gets no effect (compost: none; supplemented sawdust: one King oyster row)")
T_OPT = float(book["temp_c"].median())
print(f"cardinal rule: optimum {T_OPT:.2f} C (median of the Pleurotus book optima), limits {T_LIMITS} C")

## 2. Choose the model

All fits are ordinary least squares on log(days). Each candidate is scored by **leave one source out**: all rows of one book (or one paper) are hidden, the model is fitted on the rest, and the error in days on the hidden rows is the score (mean absolute error). The baseline is the 2.1 idea on the same training rows: the group's median typical days.

Temperature term, two forms:
- **distance**: °C away from the group's optimum (median temperature of the group's rows), one coefficient.
- **cardinal**: relative growth rate 1 at the optimum, straight lines down to the limits (the rule the app's pipeline uses today); the term is -log(rate), so a coefficient of 1 means days = typical / rate.

If two forms score within `TIE_TOLERANCE`, the simpler one (distance) is chosen.

In [ ]:
def loso(df, **fit_kwargs):
    """Leave one source out. Returns the model's predicted typical days and the fitted models."""
    pred = pd.Series(np.nan, index=df.index)
    models = {}
    for src in df["source"].unique():
        test, train = df[df["source"] == src], df[df["source"] != src]
        model = ce.fit_model(train, **fit_kwargs)
        pred[test.index] = ce.predict_typical(model, test)
        models[src] = model
    return pred, models


def loso_baseline(df, book_only=False):
    """The same leave-one-source-out split for the group-median baseline (typical days).
    book_only=False: the median over the same training rows the model sees (guide values and measurements),
    the fair test of the substrate and temperature terms. book_only=True: the 2.1 band itself (guide values only)."""
    base = pd.Series(np.nan, index=df.index)
    for src in df["source"].unique():
        test, train = df[df["source"] == src], df[df["source"] != src]
        band = ce.baseline_band(train[train["kind"] == "book"] if book_only else train)
        base[test.index] = test["plot_group"].map(lambda g: band.get(g, band["all"])["typ"])
    return base


def mae(pred, df):
    return float((pred - df["days_typ"]).abs().mean())


FIT_ARGS = dict(t_opt=T_OPT, t_limits=T_LIMITS)
temp_runs = {form: loso(data, temp_form=form, **FIT_ARGS) for form in ce.TEMP_FORMS}
base_pred = loso_baseline(data)
base21_pred = loso_baseline(data, book_only=True)
temp_scores = pd.DataFrame({
    "model MAE (days)": {f_: mae(r[0], data) for f_, r in temp_runs.items()},
}).rename(index={"none": "none (no temperature term)"})
temp_scores.loc["baseline, same training rows"] = mae(base_pred, data)
temp_scores.loc["2.1 band (guide values only)"] = mae(base21_pred, data)
display(temp_scores.round(2))

eligible = {f_: mae(temp_runs[f_][0], data) for f_ in ("distance", "cardinal")}
best = min(eligible, key=eligible.get)
TEMP_FORM = "distance" if eligible["distance"] <= eligible[best] * (1 + TIE_TOLERANCE) else best
print(f"temperature form: {TEMP_FORM} (distance {eligible['distance']:.2f}, cardinal {eligible['cardinal']:.2f} days; tie tolerance {TIE_TOLERANCE:.0%})")
# how fragile is the cardinal score? the app's pipeline uses an optimum of 25 C instead of 24.25
alt = loso(data, temp_form="cardinal", t_opt=25.0, t_limits=T_LIMITS)[0]
print(f"cardinal with the optimum at 25 C instead of {T_OPT:.2f} C: {mae(alt, data):.2f} days (a 0.75 C move changes the score by {abs(mae(alt, data) - eligible['cardinal']):.2f} days)")

In [ ]:
# C:N: fit with and without; keep only if it clearly helps (rows missing C:N: none)
assert data["cn"].notna().all()
with_cn = loso(data, temp_form=TEMP_FORM, use_cn=True, **FIT_ARGS)
no_cn = temp_runs[TEMP_FORM]
e_no, e_cn = mae(no_cn[0], data), mae(with_cn[0], data)
USE_CN = e_cn <= e_no * (1 - CN_MIN_GAIN)
print(f"error without C:N {e_no:.2f} days, with C:N {e_cn:.2f} days ({(e_cn / e_no - 1):+.0%}); needs {-CN_MIN_GAIN:.0%} or better")
print("C:N kept" if USE_CN else "C:N dropped: it does not clearly help, so the model uses group, substrate group and temperature only")
FIT_ARGS["use_cn"] = USE_CN

## 3. Coefficients as plain % effects, and the band

The final model is fitted on all rows. Each effect is the % change in days against the reference (Cool oyster, straw). The range column shows the lowest and highest value over the leave-one-source-out fits: a wide range means one source decides the number.

In [ ]:
final = ce.fit_model(data, temp_form=TEMP_FORM, **FIT_ARGS)
pred_all, fold_models = loso(data, temp_form=TEMP_FORM, **FIT_ARGS)
effects = ce.percent_effects(final)
spread = {t: [ce.percent_effects(m_).get(t) for m_ in fold_models.values()] for t in effects}
rows = []
for t, pct in effects.items():
    vals = [v for v in spread[t] if v is not None]
    name = t.replace("group=", "").replace("substrate=", "")
    rows.append({"effect": name, "vs": "Cool oyster" if t.startswith("group") else "straw", "% days": round(pct),
                 "lowest over folds": round(min(vals)), "highest over folds": round(max(vals)),
                 "plain": f"{name} {pct:+.0f}% days"})
display(pd.DataFrame(rows))
if final["rank_deficient"]:
    print("King oyster and supplemented sawdust rest on the same single row (Stamets, P. eryngii), so the data cannot tell them apart: "
          "their effects are an even split of one number. Read them as one effect, not two.")
print("intercept: Cool oyster on straw at the optimum =", round(float(np.exp(final["intercept"])), 1), "typical days")
if "temp" in final["coef"]:
    c = final["coef"]["temp"]
    if TEMP_FORM == "distance":
        print(f"temperature: each C away from the group's optimum {np.expm1(c) * 100:+.0f}% days "
              f"(optimum per group: { {g: round(v, 1) for g, v in final['group_opt'].items()} })")
    else:
        lo, hi = final["t_limits"]
        t = np.array([15, 18, 20, 22, 25, 28, 30, 32])
        eff = np.exp(c * -np.log(ce.cardinal_rate(t, final["t_opt"], lo, hi))) - 1
        print(f"temperature coefficient {c:.2f} (1.0 would be days = typical / rate); % days against the optimum:")
        print(pd.Series((eff * 100).round(), index=[f"{x} C" for x in t]).to_string())
    print("the data cover", data["temp_c"].min(), "to", data["temp_c"].max(), "C; outside that the term is an extrapolation")

In [ ]:
ratios = ce.band_ratios(data)
print(f"band ratios from {ratios['n_rows']} guide rows with min, typical and max: "
      f"min = {ratios['min_over_typ']:.2f} x typical, max = {ratios['max_over_typ']:.2f} x typical")
print("measured rows give one number each (no min or max), so they do not enter the ratios")

## 4. Check against the 2.1 baseline

Error in days, leave one source out, on the final model. Two baselines: **same rows** (the group median over the same training rows as the model, so it tests only what substrate and temperature add) and the **2.1 band** itself (group median of the guide values only). Three views: all rows, the guide values (book) and the measurements (our real bags).

Rule for the verdict (set after a first look at the numbers, so judge it yourself): the model counts as better only if it beats the *same rows* baseline on all three views. A model that wins only on the measurements, or only on the books, is not a safe replacement for the band, and beating a baseline that never saw the measurements proves nothing about substrate or temperature.

In [ ]:
err_model = (pred_all - data["days_typ"]).abs()
err_base = (base_pred - data["days_typ"]).abs()
err_21 = (base21_pred - data["days_typ"]).abs()
view = {"all rows": data.index, "guide values (book)": data.index[data.kind == "book"], "measurements": data.index[data.kind == "measured"]}
check = pd.DataFrame({k: {"rows": len(i), "model MAE (days)": err_model[i].mean(), "baseline, same rows MAE (days)": err_base[i].mean(),
                         "2.1 band MAE (days)": err_21[i].mean(), "model wins on rows (same rows)": int((err_model[i] < err_base[i]).sum())}
                      for k, i in view.items()}).T
display(check.round(2))
per_source = pd.DataFrame({"model": err_model, "baseline, same rows": err_base, "2.1 band": err_21, "source": data["source"]}).groupby("source").mean().round(1)
per_source["better than same-rows baseline"] = np.where(per_source["model"] < per_source["baseline, same rows"], "model", "baseline")
display(per_source)

# the same check without the measured sets that used less than 5% spawn (spawn is not an input, so their days carry its effect)
low_spawn = measured.loc[measured["spawn_pct"] < ce.SPAWN_RANGE[0], "source"].unique()
sub = data[~data["source"].isin(low_spawn)].reset_index(drop=True)
sp_runs = {form: loso(sub, temp_form=form, **FIT_ARGS) for form in ("none", "distance", "cardinal")}
print(f"without {list(low_spawn)} (spawn below {ce.SPAWN_RANGE[0]:g}% of the mix): baseline {mae(loso_baseline(sub), sub):.2f} days; model "
      + ", ".join(f"{k} {mae(r[0], sub):.2f}" for k, r in sp_runs.items()))

In [ ]:
wins_all = all(check.loc[k, "model MAE (days)"] < check.loc[k, "baseline, same rows MAE (days)"] for k in view)
USE_MODEL = bool(wins_all)
a = check.loc["all rows"]
print(f"Model error {a['model MAE (days)']:.1f} days; baseline on the same rows {a['baseline, same rows MAE (days)']:.1f} days; "
      f"2.1 band (guide values only) {a['2.1 band MAE (days)']:.1f} days (leave one source out).")
if USE_MODEL:
    print("VERDICT: the model is BETTER than the baseline on all rows, on the guide values and on the measurements. The app can use it.")
else:
    worse = [k for k in view if check.loc[k, "model MAE (days)"] >= check.loc[k, "baseline, same rows MAE (days)"]]
    print(f"VERDICT: the model is NOT better than the baseline (not better on: {', '.join(worse)}). The app keeps the 2.1 band.")

## 5. History

Learning curve by source, as for the other fitted models: the model (and the same-rows baseline) is fitted on 1, 2, ... other whole sources and scored on the hidden source. `train_score` is the error on the training sources, `val_score` on the hidden source, in days (lower is better). CSV and PNG go to `Desktop/MycoMaterials/kaggle/history/`.

In [ ]:
from mycodemo.config import HISTORY_DIR
from mycodemo.modeling.curves import grouped_learning_curve
from mycodemo.plot_history import plot_runs, save_history


def model_error(train, test):
    return mae(ce.predict_typical(ce.fit_model(train, temp_form=TEMP_FORM, **FIT_ARGS), test), test)


def baseline_error(train, test):
    band = ce.baseline_band(train)
    return mae(test["plot_group"].map(lambda g: band.get(g, band["all"])["typ"]), test)


common = dict(notebook="2.2.colonisation_time_estimate", train_set="other sources, whole sources added 1, 2, ...",
              test_set="the left-out source", date=datetime.date.today().isoformat())
paths = []
for run, fn, what in [("model", model_error, f"regression: group + substrate group + {TEMP_FORM} temperature" + (" + C:N" if USE_CN else "")),
                      ("baseline", baseline_error, "baseline: group median typical days over the same training rows")]:
    hist = grouped_learning_curve(data, "source", fn)
    csv, png = save_history(hist, HISTORY_DIR, "leaveonesourceout", model="colonisation_time_estimate", run=run, **common,
                            settings={"rule": what, "cv": "leave one source out", "repeats": 5, "score": "mean absolute error, days (lower is better)"})
    paths.append(csv)
    print(csv)
print(plot_runs(paths, out_png=paths[0].with_name(paths[0].stem + "_overlay.png")))
hist.round(2)

## 6. Output for the app

`mycodemo.growth.colonisation_estimate.estimate_days(group, substrate_group, temp_c, spawn_pct_of_mix)` returns `(min_days, typical_days, max_days, spawn_warning, source)`. It reads a coefficients file; if `use_model` in the file is false it returns the group's 2.1 band, and without the file it returns the usemycro 10-14 day range the app uses today.

Proposed file (nothing is written until `WRITE_COEFFICIENTS` is set to True, which waits for a yes):

In [ ]:
baseline = ce.baseline_band(pd.concat([book, usemycro]))     # the 2.1 band: guide values; usemycro rows add min and max days
coefficients = {
    "notebook": "2.2.colonisation_time_estimate", "date": datetime.date.today().isoformat(),
    "use_model": USE_MODEL,
    "verdict": ("model better than the baseline on all rows, guide values and measurements" if USE_MODEL
                else "model not better than the baseline (leave one source out): the app keeps the 2.1 band"),
    "check": {"model_mae_days": round(float(err_model.mean()), 2), "baseline_same_rows_mae_days": round(float(err_base.mean()), 2),
              "band_21_mae_days": round(float(err_21.mean()), 2),
              "n_rows": int(len(data)), "n_sources": int(data["source"].nunique())},
    "spawn_range_pct_of_mix": list(ce.SPAWN_RANGE),
    "baseline": {g: {k: round(v, 1) for k, v in b.items()} for g, b in baseline.items()},
    "band_ratios": {k: round(v, 3) if isinstance(v, float) else v for k, v in ratios.items()},
    "model": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in final.items() if k != "coef"} | {"coef": {t: round(c, 4) for t, c in final["coef"].items()}},
}
text = json.dumps(coefficients, indent=2)
print(ce.COEFFICIENTS_PATH)
print(text)
if WRITE_COEFFICIENTS:
    ce.COEFFICIENTS_PATH.write_text(text + "\n")
    print("written")

In [ ]:
# what the app function returns for the proposed file (nothing is read from disk here)
loaded = json.loads(text)
demo = pd.DataFrame([(g, sub_, t, sp_) + tuple(ce.estimate_days(g, sub_, t, sp_, loaded))
                     for g, sub_, t, sp_ in [("Cool oyster", "straw", 25, 10), ("Cool oyster", "cotton waste", 21, 3),
                                             ("Tropical oyster", "straw", 28, 12), ("King oyster", "supplemented sawdust", 24, 25)]],
                    columns=["group", "substrate", "temp C", "spawn % of mix", "min", "typical", "max", "spawn warning", "source"])
demo[["min", "typical", "max"]] = demo[["min", "typical", "max"]].round(1)
forced = {**loaded, "use_model": True}
demo["model, if it were used (typical)"] = [round(ce.estimate_days(g, sub_, t, None, forced).typical_days, 1) for g, sub_, t in zip(demo["group"], demo["substrate"], demo["temp C"])]
demo

In [ ]:
# plausibility: usemycro day ranges against the model and the band, per group (substrate straw, group optimum)
rows = []
for g, d in usemycro.groupby("plot_group"):
    lo, hi = d["days_min"].min(), d["days_max"].max()
    mod = ce.estimate_days(g, "straw", None, None, {**loaded, "use_model": True})
    rows.append({"group": g, "usemycro min": lo, "usemycro max": hi, "model min": round(mod.min_days, 1),
                 "model typical": round(mod.typical_days, 1), "model max": round(mod.max_days, 1),
                 "2.1 band": f"{baseline[g]['min']:g}-{baseline[g]['typ']:g}-{baseline[g]['max']:g}" if g in baseline else ""})
pd.DataFrame(rows)